<a href="https://colab.research.google.com/github/Andorryu/KU_EECS_836_Final_Project/blob/Task2/Task2/Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pip install

In [ ]:
!pip install pyhealth

Authenticate with colab

In [ ]:
from google.colab import auth
auth.authenticate_user()

Imports

In [ ]:
from google.cloud import bigquery
import tempfile
from pyhealth.datasets import MIMIC3Dataset
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import RNN, RETAIN
from pyhealth.tasks import MortalityPredictionMIMIC3
from pyhealth.trainer import Trainer
from pyhealth.utils import set_seed
from pathlib import Path

BigQuery queries

In [ ]:
client = bigquery.Client(project="ml-project-492918")
root = "content/mimiciii"

# Create a single folder
Path("content").mkdir(exist_ok=True)

# Create nested folders (e.g., "parent/child")
Path("content/mimiciii").mkdir(parents=True, exist_ok=True)

W = 24 # number of hours for input window

# Queries
diag_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
"""
proc_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.procedures_icd`
"""
pres_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.prescriptions`
"""
ad_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.admissions`
"""
pat_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.patients`
"""
icu_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.icustays`
"""
lab_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.labevents`
"""
dlab_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.d_labitems`
"""

client.query(diag_query).to_dataframe().to_csv(f"{root}/DIAGNOSES_ICD.csv")
client.query(proc_query).to_dataframe().to_csv(f"{root}/PROCEDURES_ICD.csv")
client.query(pres_query).to_dataframe().to_csv(f"{root}/PRESCRIPTIONS.csv")
client.query(ad_query).to_dataframe().to_csv(f"{root}/ADMISSIONS.csv")
client.query(pat_query).to_dataframe().to_csv(f"{root}/PATIENTS.csv")
client.query(icu_query).to_dataframe().to_csv(f"{root}/ICUSTAYS.csv")
client.query(lab_query).to_dataframe().to_csv(f"{root}/LABEVENTS.csv")
client.query(dlab_query).to_dataframe().to_csv(f"{root}/D_LABITEMS.csv")

Load base dataset

In [ ]:
# STEP 1: load data
base_dataset = MIMIC3Dataset(
    root=f"{root}",
    tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
    cache_dir=tempfile.TemporaryDirectory().name,
    dev=True,
)
base_dataset.stats()

Create new task for MIMIC-III timeseries mortality prediction (based off of the example MIMIC-4 timeseries prediction). This must be timeseries data so that the restriction of first W hours of data after admission time can be applied.

In [ ]:

from pyhealth.data import Patient
from datetime import datetime, timedelta
from pyhealth.tasks import BaseTask
import polars as pl

# Custom time-series task
class TimeSeriesMortality(BaseTask):
  task_name: str = "TimeSeriesMortality"

  input_schema: dict[str, str] = {
      'labs': 'timeseries'
  }
  output_schema: dict[str, str] = {
      'mortality': 'binary'
  }
  # Organize lab items by category
  LAB_CATEGORIES: dict[str, dict[str, list[str]]] = {
    "Electrolytes & Metabolic": {
      "Sodium": ["50824", "52455", "50983", "52623"],
      "Potassium": ["50822", "52452", "50971", "52610"],
      "Chloride": ["50806", "52434", "50902", "52535"],
      "Bicarbonate": ["50803", "50804"],
      "Glucose": ["50809", "52027", "50931", "52569"],
      "Calcium": ["50808", "51624"],
      "Magnesium": ["50960"],
      "Anion Gap": ["50868", "52500"],
      "Osmolality": ["52031", "50964", "51701"],
      "Phosphate": ["50970"],
    },
  }
  # Create flat list of all lab items for use in the function
  LABITEMS: list[str] = [
    item
    for category in LAB_CATEGORIES.values()
    for subcategory in category.values()
    for item in subcategory
  ]

  def __call__(self, patient: Patient) -> list[dict[str, any]]:
    input_window_hours = 24 # number of hours after admission time (W)
    samples = []

    for admission in patient.get_events(event_type="admissions"):
      admission_dischtime = datetime.strptime(
        admission.dischtime, "%Y-%m-%d %H:%M:%S"
      )
      duration_hour = (
        admission_dischtime - admission.timestamp
      ).total_seconds() / 3600
      if duration_hour <= input_window_hours:
        continue
      predict_time = admission.timestamp + timedelta(hours=input_window_hours)

      labevents_df = patient.get_events(
        event_type="labevents",
        start=admission.timestamp,
        end=predict_time,
        return_df=True,
      )
      labevents_df = labevents_df.with_columns(
        pl.col("labevents/valuenum").cast(pl.Float64, strict=False)
      )
      labevents_df = labevents_df.filter(
        pl.col("labevents/valuenum").is_not_null()
      )
      labevents_df = labevents_df.filter(
        pl.col("labevents/itemid").is_in(self.LABITEMS)
      )
      if labevents_df.height == 0:
        continue

      labevents_df = labevents_df.select(
        pl.col("timestamp"),
        pl.col("labevents/itemid"),
        pl.col("labevents/valuenum").cast(pl.Float64),
      )
      labevents_df = labevents_df.pivot(
        index="timestamp",
        columns="labevents/itemid",
        values="labevents/valuenum",
        # in case of multiple values for the same timestamp
        aggregate_function="first",
      )
      labevents_df = labevents_df.sort("timestamp")

      # Add missing columns with NaN values
      existing_cols = set(labevents_df.columns) - {"timestamp"}
      missing_cols = [item for item in self.LABITEMS if item not in existing_cols]
      for col in missing_cols:
        labevents_df = labevents_df.with_columns(pl.lit(None).alias(col))

      # Reorder columns by LABITEMS
      labevents_df = labevents_df.select("timestamp", *self.LABITEMS)

      timestamps = labevents_df["timestamp"].to_list()
      lab_values = labevents_df.drop("timestamp").to_numpy()

      mortality = int(admission.hospital_expire_flag)
      samples.append(
        {
          "patient_id": patient.patient_id,
          "admission_id": admission.hadm_id,
          "labs": (timestamps, lab_values),
          "mortality": mortality,
        }
      )

    return samples


Set task, split dataset, define model, train, and evaluate

In [ ]:
set_seed(42)

# STEP 2: set task
task = TimeSeriesMortality()
sample_dataset = base_dataset.set_task(task)
# print(sample_dataset[0])

train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

# # STEP 3: define model
model = RNN(
    dataset=sample_dataset,
)

# # STEP 4: define trainer
trainer = Trainer(model=model)
trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=1,
    monitor="roc_auc",
)

# # STEP 5: evaluate
print(trainer.evaluate(test_dataloader))